# Analytics Agent Backend - Google Cloud Shell Setup

FastAPI backend ko Google Cloud Shell pe run karne ke liye.

**Features:**
- Public URL via Cloudflare Tunnel
- Free Cloud Shell environment
- Auto-reconnect keep-alive
- Test API endpoint instantly

## Step 1: Clone/Upload Backend Code

**Option A: Git clone**

In [ ]:
# If your repo is on GitHub
# !git clone https://github.com/YOUR_USERNAME/analytics-agent.git
# %cd analytics-agent/backend

# Or manually upload files
print("📁 Make sure you're in the backend directory")
!pwd

## Step 2: Install Dependencies

In [ ]:
# Install Python dependencies
!pip install -q fastapi uvicorn[standard] pydantic pydantic-settings httpx PyJWT \
    pandas numpy scipy statsmodels scikit-learn matplotlib openpyxl xlrd \
    python-multipart cryptography psycopg[binary] pyarrow openai

print("✅ Dependencies installed")

## Step 3: Create .env File

**IMPORTANT:** Cloud Shell me `.env` file already hai? Skip karo.

Nahi hai toh create karo:

In [ ]:
import os
from pathlib import Path

# Check if .env exists
env_file = Path('.env')

if env_file.exists():
    print("✅ .env file already exists")
    !head -5 .env
else:
    print("⚠️ .env file not found!")
    print("\nOption 1: Upload .env file via Cloud Shell editor")
    print("Option 2: Run this to create from .env.example:")
    print("   cp .env.example .env")
    print("   Then edit with: cloudshell edit .env")

## Step 4: Install Cloudflared

In [ ]:
import subprocess
import os

# Check if cloudflared already installed
result = subprocess.run(['which', 'cloudflared'], capture_output=True)

if result.returncode == 0:
    print("✅ cloudflared already installed")
else:
    print("📦 Installing cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared-linux-amd64
    !sudo mv cloudflared-linux-amd64 /usr/local/bin/cloudflared
    print("✅ cloudflared installed")

## Step 5: Start Backend Server

**⚠️ IMPORTANT:** Is cell ko run karne ke baad output me public URL aayega.

Cloud Shell 60 min idle timeout hai, toh tab khula rakhna.

In [ ]:
import threading
import subprocess
import time
import re
from IPython.display import clear_output, HTML, display

# Start uvicorn in background
def start_uvicorn():
    import uvicorn
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000, log_level="info")

print("🚀 Starting FastAPI server...")
uvicorn_thread = threading.Thread(target=start_uvicorn, daemon=True)
uvicorn_thread.start()

time.sleep(5)
print("✅ FastAPI running on http://localhost:8000")

# Start cloudflared tunnel
print("\n🌐 Starting Cloudflare tunnel...\n")
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

# Extract public URL
public_url = None
max_attempts = 20
attempt = 0

for line in tunnel_process.stdout:
    print(line.strip())
    
    # Look for trycloudflare.com URL
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    
    attempt += 1
    if attempt > max_attempts:
        break

# Display results
if public_url:
    clear_output(wait=True)
    display(HTML(f'''
    <div style="padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; margin: 10px 0;">
        <h2 style="color: #2e7d32; margin: 0 0 10px 0;">🚀 Backend is LIVE!</h2>
        <p style="margin: 5px 0;"><strong>Public URL:</strong></p>
        <p style="margin: 5px 0; font-family: monospace; background: white; padding: 10px; border-radius: 4px; font-size: 14px;">
            <a href="{public_url}" target="_blank" style="color: #1976d2; text-decoration: none;">{public_url}</a>
        </p>
        <p style="margin: 15px 0 5px 0;"><strong>API Docs:</strong></p>
        <p style="margin: 5px 0; font-family: monospace; background: white; padding: 10px; border-radius: 4px; font-size: 14px;">
            <a href="{public_url}/docs" target="_blank" style="color: #1976d2; text-decoration: none;">{public_url}/docs</a>
        </p>
        <hr style="margin: 20px 0; border: none; border-top: 1px solid #ccc;">
        <p style="margin: 10px 0 5px 0; color: #d32f2f; font-weight: bold;">⚠️ Android App Configuration:</p>
        <p style="margin: 5px 0; font-family: monospace; background: #fff3e0; padding: 10px; border-radius: 4px; font-size: 13px;">
            ANALYTICS_AGENT_API_URL={public_url}
        </p>
        <p style="margin: 15px 0 5px 0; color: #666; font-size: 13px;">
            💡 Is URL ko Android app ke <code>gradle.properties</code> ya GitHub Secrets me daalo
        </p>
    </div>
    '''))
    
    print(f"\n" + "="*70)
    print(f"✅ Backend URL: {public_url}")
    print(f"📖 API Docs: {public_url}/docs")
    print(f"\n💡 Android config me daalo:")
    print(f"   ANALYTICS_AGENT_API_URL={public_url}")
    print("="*70)
    print(f"\n⚠️ Cloud Shell 60 min idle timeout - tab khula rakhein")
    print(f"\n🔄 Keep-alive: Run the next cell")
else:
    print("\n⚠️ Could not extract public URL from tunnel logs.")
    print("\nTroubleshooting:")
    print("1. Check if port 8000 is free: lsof -i :8000")
    print("2. Check FastAPI logs above")
    print("3. Manually run: cloudflared tunnel --url http://localhost:8000")

## Step 6: Keep-Alive (Cloud Shell Timeout Prevention)

Cloud Shell 60 minutes idle timeout hai. Keep-alive chalane ke liye:

In [ ]:
%%javascript
// Keep session alive by clicking connect button
function KeepAlive() {
    console.log("⏰ Keep-alive ping");
    // Trigger activity
    const evt = new MouseEvent('mousemove');
    document.dispatchEvent(evt);
}

// Ping every 50 seconds (before 60 sec timeout)
setInterval(KeepAlive, 50000);
console.log("✅ Keep-alive started (pings every 50s)");

## Step 7: Test API (Optional)

In [ ]:
import requests

try:
    # Test health endpoint
    response = requests.get('http://localhost:8000/health', timeout=5)
    print(f"✅ Status: {response.status_code}")
    print(f"Response: {response.json()}")
except Exception as e:
    print(f"❌ Error: {e}")

---

## 📝 Important Notes:

### ⏰ Cloud Shell Limitations:
- **60 min idle timeout** - activity karte raho ya keep-alive chalaao
- **Session expires** - naya session = naya URL (Android config update karna padega)
- **5 GB persistent storage** - `/home/username` me files save rahti hain

### 🔄 Session Restart:
1. Cloud Shell reconnect hone par notebook reopen karo
2. Step 5 cell run karo (dependencies already installed rahenge)
3. Naya public URL milega

### ✅ Verification:
```bash
# Check if backend running
curl http://localhost:8000/health

# Check tunnel status
ps aux | grep cloudflared

# View logs
tail -f /tmp/uvicorn.log
```

### 🚀 Production Alternative:
For permanent URL without timeouts:
- **Google Cloud Run** (use `deploy-gcloud.sh`)
- **Railway** ($5-10/month)
- **Render** (free tier with limitations)

---

## 🆘 Troubleshooting:

**Problem: Port already in use**
```bash
# Find and kill process
lsof -ti:8000 | xargs kill -9
```

**Problem: .env not loaded**
```bash
# Verify env file
cat .env | head -5
```

**Problem: Module not found**
```bash
# Reinstall dependencies
pip install -r requirements.txt --force-reinstall
```